In [1]:
import pandas as pd
import numpy as np
import random
import time
import json
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import BertTokenizer, BertModel 
from transformers import logging

In [2]:
batch_size = 64
historical_vector_nums = 20
entities_word_nums = 30
vector_embed_dims = 256
entities_embed_dims = 100
bert_encoding_length = 200
device = 'cuda'

# Training

In [8]:
news = pd.read_csv('train/train_news.tsv', delimiter='\t', header=None)
news.columns = ['news_id', 'category', 'subcategory', 'title', 'abstract', 'URL', 'title_entities', 'abstract_entities']

class BertDataset(Dataset):
    def __init__(self, news, tokenizer):
        self.news = news
        self.tokenizer = tokenizer
    
    def __len__(self):
        return len(self.news)
    
    def __getitem__(self, index):
        news_id, category, subcategory, title, abstract, _, _, _ = self.news.iloc[index]
        
        title = "" if isinstance(title, float) else title
        abstract = "" if isinstance(abstract, float) else abstract
        encoding = self.tokenizer(
            text = category + " " + subcategory,
            text_pair = title + " " + abstract,
            return_tensors = "pt",
            padding = "max_length",
            truncation = True,
            max_length = bert_encoding_length
        )
        return news_id, encoding.input_ids[0], encoding.token_type_ids[0], encoding.attention_mask[0]

logging.set_verbosity_error()
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert = BertModel.from_pretrained('bert-base-uncased')
bert = bert.to(device)

bert_dataset = BertDataset(news, tokenizer)
bert_loader = DataLoader(bert_dataset, batch_size=batch_size, shuffle=False)

bert_embedding = {}
for ids, input_ids, token_type_ids, attention_mask in tqdm(bert_loader):
    input_ids = input_ids.to(device)
    token_type_ids = token_type_ids.to(device)
    attention_mask = attention_mask.to(device)
    
    with torch.no_grad():
        bert_output = bert(
            input_ids = input_ids,
            token_type_ids = token_type_ids,
            attention_mask = attention_mask
        )
    
    for id_, output in zip(ids, bert_output[1]):
        bert_embedding[id_] = output

with open('train/train_bert_embedding.vec', 'w') as file:
    for key, values in tqdm(bert_embedding.items()):
        values = [round(v.item(), 6) for v in values]
        line = f"{key} {' '.join(map(str, values))}\n"
        file.write(line)

100%|██████████| 1587/1587 [11:24<00:00,  2.32it/s]


In [3]:
behaviors = pd.read_csv('train/train_behaviors.tsv', delimiter='\t', index_col=0, header=None)
behaviors.columns = ['user', 'time', 'clicked_news', 'impressions']
behaviors = behaviors.sample(frac=1, random_state=42)
train_behaviors = behaviors[:int(len(behaviors) * 0.8)]
valid_behaviors = behaviors[int(len(behaviors) * 0.8):]

news = pd.read_csv('train/train_news.tsv', delimiter='\t', header=None)
news.columns = ['news_id', 'category', 'subcategory', 'title', 'abstract', 'URL', 'title_entities', 'abstract_entities']
news_dict = {data['news_id']: data.iloc[1:] for _, data in news.iterrows()}

bert_embedding = {}
f = open('train/train_bert_embedding.vec')
for line in f:
    values = line.split()
    word = values[0]
    coefs = np.asarray(values[1:], dtype='float32')
    bert_embedding[word] = coefs
f.close()

entity_embedding = {}
f = open('train/train_entity_embedding.vec')
for line in f:
    values = line.split()
    word = values[0]
    coefs = np.asarray(values[1:], dtype='float32')
    entity_embedding[word] = coefs
f.close()

In [4]:
class RecommendationDataset(Dataset):
    def __init__(self, behaviors, news_dict, bert_embedding, entity_embedding, mode='train'):
        self.behaviors = behaviors
        self.news_dict = news_dict
        self.bert_embedding = bert_embedding
        self.entity_embedding = entity_embedding
        self.mode = mode
        
    def __len__(self):
        return len(self.behaviors)

    def padding(self, item, size):
        item = torch.stack(item)[:size]
        if item.size(0) < size:
            padding_length = size - item.size(0)
            padding_item = torch.zeros((padding_length, *item.shape[1:]))
            item = torch.cat((item, padding_item), dim=0)
        return item
    
    def extract_entities(self, news):
        _, _, _, _, _, title_entities, abstract_entities = self.news_dict[news]
        
        title_entities = "[]" if isinstance(title_entities, float) else title_entities
        abstract_entities = "[]" if isinstance(abstract_entities, float) else abstract_entities
        
        vector = []
        entities = json.loads(title_entities)
        ids = [entity['WikidataId'] for entity in entities]
        vector += [torch.tensor(self.entity_embedding[id_]) for id_ in ids if id_ in self.entity_embedding]
        entities = json.loads(abstract_entities)
        ids = [entity['WikidataId'] for entity in entities]
        vector += [torch.tensor(self.entity_embedding[id_]) for id_ in ids if id_ in self.entity_embedding]

        if len(vector) == 0:
            vector = torch.zeros((entities_word_nums, entities_embed_dims))
        else:
            vector = self.padding(vector, entities_word_nums)
                
        return vector
    
    def __getitem__(self, index):
        _, _, clicked_news, impressions = self.behaviors.iloc[index]
        
        clicked_news = clicked_news.split()
        history_bert_vectors = [torch.tensor(self.bert_embedding[news]) for news in clicked_news]
        history_bert_vectors = self.padding(history_bert_vectors, historical_vector_nums)
        history_entity_vectors = [self.extract_entities(news) for news in clicked_news]
        history_entity_vectors = self.padding(history_entity_vectors, historical_vector_nums)

        impressions = impressions.split()
        impression_news = [impression.split('-')[0] for impression in impressions]
        recommen_bert_vectors = [torch.tensor(self.bert_embedding[news]) for news in impression_news]
        recommen_bert_vectors = torch.stack(recommen_bert_vectors)
        recommen_entity_vectors = [self.extract_entities(news) for news in impression_news]
        recommen_entity_vectors = torch.stack(recommen_entity_vectors)

        packed = {
            'history_bert_vectors':    history_bert_vectors,
            'history_entity_vectors':   history_entity_vectors,
            'recommen_bert_vectors':    recommen_bert_vectors,
            'recommen_entity_vectors':  recommen_entity_vectors
        }
        
        if self.mode == 'train':
            labels = torch.tensor([int(impression.split('-')[1]) for impression in impressions])
            return packed, labels
        elif self.mode == 'test':
            return packed
        
train_dataset = RecommendationDataset(train_behaviors, news_dict, bert_embedding, entity_embedding)
valid_dataset = RecommendationDataset(valid_behaviors, news_dict, bert_embedding, entity_embedding)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)

In [5]:
class NewsVectorEncoder(nn.Module):
    def __init__(self):
        super(NewsVectorEncoder, self).__init__()
        if (vector_embed_dims % 2) != 0 :
            raise ValueError("news_embed_dims must be even number")
        
        self.bert_fc = nn.Linear(768, vector_embed_dims // 2)
        self.entity_fc = nn.Linear(entities_word_nums*entities_embed_dims, vector_embed_dims // 2)
    
    def forward(self, bert_vectors, entity_vectors):
        bert_out = self.bert_fc(bert_vectors)
        entity_out = torch.flatten(entity_vectors, -2, -1)
        entity_out = self.entity_fc(entity_out)
        
        return torch.cat((bert_out, entity_out), dim=-1)
        
class UserVectorEncoder(nn.Module):
    def __init__(self):
        super(UserVectorEncoder, self).__init__()
        self.encoder = nn.Linear(historical_vector_nums*vector_embed_dims, vector_embed_dims)
    
    def forward(self, x):
        return self.encoder(x.flatten(-2, -1))
        
class RecommendationModel(nn.Module):
    def __init__(self):
        super(RecommendationModel, self).__init__()
        
        self.nve = NewsVectorEncoder()
        self.uve = UserVectorEncoder()
    
    def forward(self, history_bert_vectors, history_entity_vectors, recommen_bert_vectors, recommen_entity_vectors):
    
        history_lens = history_entity_vectors.shape[1]
        history_news_vectors = []
        for i in range(history_lens):
            news_vector = self.nve(history_bert_vectors[:, i, :], history_entity_vectors[:, i, :, :])
            history_news_vectors.append(news_vector)
        history_news_vectors = torch.stack(history_news_vectors).permute(1, 0, 2) 
        # (Num News, Batch, Embedded Size) -> (Batch, Num News, Embedded Size)

        recommen_lens = recommen_entity_vectors.shape[1]
        recommen_news_vectors = []
        for i in range(recommen_lens):
            news_vector = self.nve(recommen_bert_vectors[:, i, :], recommen_entity_vectors[:, i, :, :])
            recommen_news_vectors.append(news_vector)
        recommen_news_vectors = torch.stack(recommen_news_vectors).permute(1, 0, 2)
        # (Num News, Batch, Embedded Size) -> (Batch, Num News, Embedded Size)
        
        user_vector = self.uve(history_news_vectors)
        attention_score = (recommen_news_vectors @ user_vector.unsqueeze(-1)).squeeze()
        return attention_score

In [ ]:
num_epoch = 5
show_freq = 10

model = RecommendationModel()
model = model.to(device)

loss = nn.MultiLabelSoftMarginLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

best_valid_auc = 0.0
for epoch in range(num_epoch):
    epoch_start_time = time.time()
    train_loss, valid_loss = 0.0, 0.0
    train_count, valid_count = 0.0, 0.0
    train_true, valid_true = [], []
    train_pred, valid_pred = [], []

    model.train()
    for i, (packed, labels) in enumerate(train_loader):
        for key, value in packed.items():
            packed[key] = value.to(device)
        labels = labels.to(device)
            
        outputs = model(**packed)
        
        batch_loss = loss(outputs, labels)
        batch_loss.backward()
        optimizer.step()
        model.zero_grad()
        
        train_loss += batch_loss.item()
        train_count += labels.shape[0] * labels.shape[1]
        train_true += labels.reshape(-1).cpu().detach().numpy().tolist()
        train_pred += torch.sigmoid(outputs.reshape(-1)).cpu().detach().numpy().tolist()
        
        if (i+1) % show_freq == 0 or (i+1) == len(train_loader):
            train_auc = roc_auc_score(train_true, train_pred)
            print('[{:02d}/{:02d} - {:04d}/{:04d}] '.format(epoch+1, num_epoch, i+1, len(train_loader))
                + '{:2.2f} sec '.format(time.time() - epoch_start_time)
                + 'Train AUC: {:3.2f} Loss: {:3.4f} '.format(train_auc, train_loss/train_count*1000)
            )
            
    model.eval()
    for i, (packed, labels) in enumerate(valid_loader):
        for key, value in packed.items():
            packed[key] = value.to(device)
        labels = labels.to(device)
        
        with torch.no_grad():
            outputs = model(**packed)
        
        batch_loss = loss(outputs, labels)
        
        valid_loss += batch_loss.item()
        valid_count += labels.shape[0] * labels.shape[1]
        valid_true += labels.reshape(-1).cpu().detach().numpy().tolist()
        valid_pred += torch.sigmoid(outputs.reshape(-1)).cpu().detach().numpy().tolist()
        
        if (i+1) % show_freq == 0 or (i+1) == len(valid_loader):
            valid_auc = roc_auc_score(valid_true, valid_pred)
            print('[{:02d}/{:02d} - {:04d}/{:04d}] '.format(epoch+1, num_epoch, i+1, len(valid_loader))
                + '{:2.2f} sec '.format(time.time() - epoch_start_time)
                + 'Valid AUC: {:3.2f} Loss: {:3.4f} '.format(valid_auc, valid_loss/valid_count*1000)
            )
            
    if best_valid_auc < valid_auc:
        best_valid_auc = valid_auc
        torch.save(model.state_dict(), 'best_weight.pth')
        
print(f"Best Validation AUC: {best_valid_auc}")

[01/05 - 0010/3567] 1.48 sec Train AUC: 0.49 Loss: 0.4734 
[01/05 - 0020/3567] 3.04 sec Train AUC: 0.49 Loss: 0.4464 
[01/05 - 0030/3567] 4.54 sec Train AUC: 0.49 Loss: 0.4266 
[01/05 - 0040/3567] 6.07 sec Train AUC: 0.50 Loss: 0.4111 
[01/05 - 0050/3567] 7.61 sec Train AUC: 0.50 Loss: 0.3999 
[01/05 - 0060/3567] 9.17 sec Train AUC: 0.51 Loss: 0.3941 
[01/05 - 0070/3567] 10.69 sec Train AUC: 0.51 Loss: 0.3892 
[01/05 - 0080/3567] 12.19 sec Train AUC: 0.52 Loss: 0.3868 
[01/05 - 0090/3567] 13.71 sec Train AUC: 0.52 Loss: 0.3849 
[01/05 - 0100/3567] 15.22 sec Train AUC: 0.52 Loss: 0.3826 
[01/05 - 0110/3567] 16.87 sec Train AUC: 0.53 Loss: 0.3813 
[01/05 - 0120/3567] 18.49 sec Train AUC: 0.53 Loss: 0.3795 
[01/05 - 0130/3567] 20.06 sec Train AUC: 0.53 Loss: 0.3786 
[01/05 - 0140/3567] 21.66 sec Train AUC: 0.53 Loss: 0.3769 
[01/05 - 0150/3567] 23.20 sec Train AUC: 0.53 Loss: 0.3767 
[01/05 - 0160/3567] 24.80 sec Train AUC: 0.54 Loss: 0.3758 
[01/05 - 0170/3567] 26.35 sec Train AUC: 0.54 

[01/05 - 1370/3567] 236.14 sec Train AUC: 0.61 Loss: 0.3580 
[01/05 - 1380/3567] 238.09 sec Train AUC: 0.61 Loss: 0.3580 
[01/05 - 1390/3567] 240.00 sec Train AUC: 0.61 Loss: 0.3580 
[01/05 - 1400/3567] 242.06 sec Train AUC: 0.61 Loss: 0.3580 
[01/05 - 1410/3567] 244.03 sec Train AUC: 0.61 Loss: 0.3579 
[01/05 - 1420/3567] 245.99 sec Train AUC: 0.61 Loss: 0.3579 
[01/05 - 1430/3567] 247.88 sec Train AUC: 0.61 Loss: 0.3577 
[01/05 - 1440/3567] 249.86 sec Train AUC: 0.61 Loss: 0.3576 
[01/05 - 1450/3567] 251.76 sec Train AUC: 0.61 Loss: 0.3576 
[01/05 - 1460/3567] 253.71 sec Train AUC: 0.61 Loss: 0.3576 
[01/05 - 1470/3567] 255.62 sec Train AUC: 0.61 Loss: 0.3576 
[01/05 - 1480/3567] 257.52 sec Train AUC: 0.61 Loss: 0.3575 
[01/05 - 1490/3567] 259.53 sec Train AUC: 0.61 Loss: 0.3575 
[01/05 - 1500/3567] 261.50 sec Train AUC: 0.61 Loss: 0.3575 
[01/05 - 1510/3567] 263.47 sec Train AUC: 0.61 Loss: 0.3575 
[01/05 - 1520/3567] 265.39 sec Train AUC: 0.61 Loss: 0.3574 
[01/05 - 1530/3567] 267.

[01/05 - 2720/3567] 529.42 sec Train AUC: 0.63 Loss: 0.3539 
[01/05 - 2730/3567] 531.85 sec Train AUC: 0.63 Loss: 0.3539 
[01/05 - 2740/3567] 534.23 sec Train AUC: 0.63 Loss: 0.3539 
[01/05 - 2750/3567] 536.57 sec Train AUC: 0.63 Loss: 0.3539 
[01/05 - 2760/3567] 539.03 sec Train AUC: 0.63 Loss: 0.3539 
[01/05 - 2770/3567] 541.50 sec Train AUC: 0.63 Loss: 0.3539 
[01/05 - 2780/3567] 543.84 sec Train AUC: 0.63 Loss: 0.3539 
[01/05 - 2790/3567] 546.26 sec Train AUC: 0.63 Loss: 0.3538 
[01/05 - 2800/3567] 548.79 sec Train AUC: 0.63 Loss: 0.3538 
[01/05 - 2810/3567] 551.26 sec Train AUC: 0.63 Loss: 0.3538 
[01/05 - 2820/3567] 553.80 sec Train AUC: 0.63 Loss: 0.3538 
[01/05 - 2830/3567] 556.33 sec Train AUC: 0.63 Loss: 0.3537 
[01/05 - 2840/3567] 558.75 sec Train AUC: 0.63 Loss: 0.3537 
[01/05 - 2850/3567] 561.22 sec Train AUC: 0.63 Loss: 0.3537 
[01/05 - 2860/3567] 563.78 sec Train AUC: 0.63 Loss: 0.3537 
[01/05 - 2870/3567] 566.29 sec Train AUC: 0.63 Loss: 0.3536 
[01/05 - 2880/3567] 568.

# Testing

In [ ]:
news = pd.read_csv('test/test_news.tsv', delimiter='\t', header=None)
news.columns = ['news_id', 'category', 'subcategory', 'title', 'abstract', 'URL', 'title_entities', 'abstract_entities']

logging.set_verbosity_error()
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert = BertModel.from_pretrained('bert-base-uncased')
bert = bert.to(device)

bert_dataset = BertDataset(news, tokenizer)
bert_loader = DataLoader(bert_dataset, batch_size=batch_size, shuffle=False)

bert_embedding = {}
for ids, input_ids, token_type_ids, attention_mask in tqdm(bert_loader):
    input_ids = input_ids.to(device)
    token_type_ids = token_type_ids.to(device)
    attention_mask = attention_mask.to(device)
    
    with torch.no_grad():
        bert_output = bert(
            input_ids = input_ids,
            token_type_ids = token_type_ids,
            attention_mask = attention_mask
        )
    
    for id_, output in zip(ids, bert_output[1]):
        bert_embedding[id_] = output

with open('test/test_bert_embedding.vec', 'w') as file:
    for key, values in tqdm(bert_embedding.items()):
        values = [round(v.item(), 6) for v in values]
        line = f"{key} {' '.join(map(str, values))}\n"
        file.write(line)

In [ ]:
test_behaviors = pd.read_csv('test/test_behaviors.tsv', delimiter='\t', index_col=0, header=None)
test_behaviors.columns = ['user', 'time', 'clicked_news', 'impressions']

news = pd.read_csv('test/test_news.tsv', delimiter='\t', header=None)
news.columns = ['news_id', 'category', 'subcategory', 'title', 'abstract', 'URL', 'title_entities', 'abstract_entities']
news_dict = {data['news_id']: data.iloc[1:] for _, data in news.iterrows()}

bert_embedding = {}
f = open('test/test_bert_embedding.vec')
for line in f:
    values = line.split()
    word = values[0]
    coefs = np.asarray(values[1:], dtype='float32')
    bert_embedding[word] = coefs
f.close()

entity_embedding = {}
f = open('test/test_entity_embedding.vec')
for line in f:
    values = line.split()
    word = values[0]
    coefs = np.asarray(values[1:], dtype='float32')
    entity_embedding[word] = coefs
f.close()

In [ ]:
test_dataset = RecommendationDataset(train_behaviors, news_dict, bert_embedding, entity_embedding, mode='test')
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

model = RecommendationModel()
model = model.to(device)


for epoch in range(num_epoch):
    epoch_start_time = time.time()
    train_loss, valid_loss = 0.0, 0.0
    train_count, valid_count = 0.0, 0.0
    train_true, valid_true = [], []
    train_pred, valid_pred = [], []

    model.train()
    for i, (packed, labels) in enumerate(train_loader):
        for key, value in packed.items():
            packed[key] = value.to(device)
        labels = labels.to(device)
            
        outputs = model(**packed)
        
        batch_loss = loss(outputs, labels)
        batch_loss.backward()
        optimizer.step()
        model.zero_grad()
        
        train_loss += batch_loss.item()
        train_count += labels.shape[0] * labels.shape[1]
        train_true += labels.reshape(-1).cpu().detach().numpy().tolist()
        train_pred += torch.sigmoid(outputs.reshape(-1)).cpu().detach().numpy().tolist()
        
        if (i+1) % show_freq == 0 or (i+1) == len(train_loader):
            train_auc = roc_auc_score(train_true, train_pred)
            print('[{:02d}/{:02d} - {:04d}/{:04d}] '.format(epoch+1, num_epoch, i+1, len(train_loader))
                + '{:2.2f} sec '.format(time.time() - epoch_start_time)
                + 'Train AUC: {:3.2f} Loss: {:3.4f} '.format(train_auc, train_loss/train_count*1000)
            )
            
    model.eval()
    for i, (packed, labels) in enumerate(valid_loader):
        for key, value in packed.items():
            packed[key] = value.to(device)
        labels = labels.to(device)
        
        with torch.no_grad():
            outputs = model(**packed)
        
        batch_loss = loss(outputs, labels)
        
        valid_loss += batch_loss.item()
        valid_count += labels.shape[0] * labels.shape[1]
        valid_true += labels.reshape(-1).cpu().detach().numpy().tolist()
        valid_pred += torch.sigmoid(outputs.reshape(-1)).cpu().detach().numpy().tolist()
        
        if (i+1) % show_freq == 0 or (i+1) == len(valid_loader):
            valid_auc = roc_auc_score(valid_true, valid_pred)
            print('[{:02d}/{:02d} - {:04d}/{:04d}] '.format(epoch+1, num_epoch, i+1, len(valid_loader))
                + '{:2.2f} sec '.format(time.time() - epoch_start_time)
                + 'Train AUC: {:3.2f} Loss: {:3.4f} '.format(valid_auc, valid_loss/valid_count*1000)
            )
            
    if best_valid_auc < valid_auc:
        best_valid_auc = valid_auc
        torch.save(model.state_dict(), 'best_weight.pth')
        
print(f"Best Validation AUC: {best_valid_auc}")